In [1]:
! pip install python-chess
import chess
!pip install torch torchvision
import numpy as np
import torch
import torch.nn as nn
import chess.pgn
import csv
from torch.utils.data import Dataset
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 61.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256=f76f66f4d4bf73143611ddacdcdb719f3337268c29cd9e2f83726c20cf6f8eac
  Stored in directory: /root/.cache/pip/wheels/83/1f/4e/8f4300f7dd554eb8de70ddfed96e94d3d030ace10c5b53d447
Successfully built chess
Using device: cuda


In [2]:
from google.colab import drive
drive.mount('/content/drive')
file_path='/content/drive/MyDrive/stockfish_position_evaluations.csv'

Mounted at /content/drive


In [3]:
board=chess.Board()
piece_to_plane = {
    (chess.PAWN, True): 0,
    (chess.KNIGHT, True): 1,
    (chess.BISHOP, True): 2,
    (chess.ROOK, True): 3,
    (chess.QUEEN, True): 4,
    (chess.KING, True): 5,
    (chess.PAWN, False): 6,
    (chess.KNIGHT, False): 7,
    (chess.BISHOP, False): 8,
    (chess.ROOK, False): 9,
    (chess.QUEEN, False): 10,
    (chess.KING, False): 11,
}
def encode_board(board):
   x = np.zeros((18, 8, 8), dtype=np.float32)

   for square, piece in board.piece_map().items():
        plane = piece_to_plane[(piece.piece_type, piece.color)]
        row = 7 - chess.square_rank(square)
        col = chess.square_file(square)
        x[plane, row, col] = 1.0
   if board.has_kingside_castling_rights(chess.WHITE):
      x[12, : , :]= 1.0
   if board.has_kingside_castling_rights(chess.BLACK):
      x[13, :, :]=1.0
   if board.has_queenside_castling_rights(chess.WHITE):
      x[14, :, :]=1.0
   if board.has_queenside_castling_rights(chess.BLACK):
      x[15, :, :]=1.0
   if board.ep_square is not None:
      row=7-chess.square_rank(board.ep_square)
      col=chess.square_file(board.ep_square)
      x[16, row, col]=1.0
   if board.turn == chess.WHITE:
      x[17, :, :] = 1.0

   return x
def encode_boards(boards):
    return np.stack([encode_board(b) for b in boards])

In [4]:
PIECE_VALUES = {
    chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3,
    chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0
}

HANGING_PIECE_WEIGHT = 0.3

def hanging_material_score(board):
    score = 0.0
    for square, piece in board.piece_map().items():
        attackers = board.attackers(not piece.color, square)
        if not attackers:
            continue
        defenders = board.attackers(piece.color, square)
        if defenders:
            continue
        value = PIECE_VALUES.get(piece.piece_type, 0) * HANGING_PIECE_WEIGHT
        score += value if piece.color == chess.BLACK else -value
    return score

KING_SAFETY_WEIGHT = 0.05

def king_safety_score(board):
    def shield_pawn_count(color):
        king_sq = board.king(color)
        if king_sq is None:
            return 0
        king_file = chess.square_file(king_sq)
        king_rank = chess.square_rank(king_sq)
        forward = 1 if color == chess.WHITE else -1
        shield_rank = king_rank + forward
        if not (0 <= shield_rank <= 7):
            return 0
        count = 0
        for f in (king_file - 1, king_file, king_file + 1):
            if 0 <= f <= 7:
                piece = board.piece_at(chess.square(f, shield_rank))
                if piece and piece.piece_type == chess.PAWN and piece.color == color:
                    count += 1
        return count

    white_shield = shield_pawn_count(chess.WHITE)
    black_shield = shield_pawn_count(chess.BLACK)
    return (white_shield - black_shield) * KING_SAFETY_WEIGHT
KNIGHT_PST = [
    -0.50, -0.40, -0.30, -0.30, -0.30, -0.30, -0.40, -0.50,
    -0.40, -0.20,  0.00,  0.00,  0.00,  0.00, -0.20, -0.40,
    -0.30,  0.00,  0.10,  0.15,  0.15,  0.10,  0.00, -0.30,
    -0.30,  0.05,  0.15,  0.20,  0.20,  0.15,  0.05, -0.30,
    -0.30,  0.00,  0.15,  0.20,  0.20,  0.15,  0.00, -0.30,
    -0.30,  0.05,  0.10,  0.15,  0.15,  0.10,  0.05, -0.30,
    -0.40, -0.20,  0.00,  0.05,  0.05,  0.00, -0.20, -0.40,
    -0.50, -0.40, -0.30, -0.30, -0.30, -0.30, -0.40, -0.50,
]

def knight_position_score(board):
    score = 0.0
    for square in board.pieces(chess.KNIGHT, chess.WHITE):
        score += KNIGHT_PST[square]
    for square in board.pieces(chess.KNIGHT, chess.BLACK):
        score -= KNIGHT_PST[chess.square_mirror(square)]
    return score

NUM_HEURISTIC_FEATURES = 3

def compute_heuristic_features(board):
    return np.array([
        hanging_material_score(board),
        king_safety_score(board),
        knight_position_score(board),
    ], dtype=np.float32)


In [5]:
class ChessValueNet(nn.Module):
     def __init__(self, num_heuristic_features=NUM_HEURISTIC_FEATURES):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(18, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(256 * 8 * 8 + num_heuristic_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )
     def forward(self, x, heuristics):
        x = self.conv(x)
        x = x.flatten(1)
        x = torch.cat([x, heuristics], dim=1)
        return self.head(x)


In [6]:
class ChessEvalCSVDataset(Dataset):
  def __init__(self, csv_path, max_positions=None):
    self.samples = []
    self.load_positions(csv_path, max_positions)

  def load_positions(self, csv_path, max_positions):
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            fen = row["fen"]
            eval_str = row["evaluation"]

            if eval_str.startswith("M") or eval_str.startswith("#"):
                mate_in = int(eval_str.lstrip("M#"))
                pos_target = 1.0 if mate_in > 0 else -1.0
            else:
                centipawns = int(eval_str)
                pos_target = max(-1.0, min(1.0, centipawns / 1000.0))

            board = chess.Board(fen)
            heuristics = compute_heuristic_features(board)
            self.samples.append((encode_board(board), heuristics, pos_target))

            if max_positions is not None and (i + 1) >= max_positions:
                break

  def __len__(self):
      return len(self.samples)

  def __getitem__(self, idx):
      x, h, y = self.samples[idx]
      return torch.tensor(x, dtype=torch.float32), torch.tensor(h, dtype=torch.float32), torch.tensor([y], dtype=torch.float32)


In [7]:
dataset = ChessEvalCSVDataset(file_path, max_positions=None)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset = Subset(dataset, range(0, train_size))
val_dataset = Subset(dataset, range(train_size, len(dataset)))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

model = ChessValueNet().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

best_val_loss = float("inf")

train_losses = []
val_losses = []

for epoch in range(20):
    model.train()
    total_train_loss = 0.0

    for x, h, y in train_loader:
        x, h, y = x.to(device), h.to(device), y.to(device)
        pred = model(x, h)
        loss = criterion(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    print("epoch", epoch, "train loss", avg_train_loss)

    model.eval()
    total_val_loss = 0.0

    with torch.no_grad():
        for x, h, y in val_loader:
            x, h, y = x.to(device), h.to(device), y.to(device)
            pred = model(x, h)
            loss = criterion(pred, y)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    print("epoch", epoch, "val loss", avg_val_loss)

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": avg_train_loss,
        "val_loss": avg_val_loss,
    }, "latest.pth")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
        }, "best.pth")

    model.train()

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(range(len(train_losses)), train_losses, label='Training Loss')
plt.plot(range(len(val_losses)), val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss Over Epochs')
plt.legend()
plt.grid(True)
plt.show()


epoch 0 train loss 0.09821665873720811
epoch 0 val loss 0.05950830657864651
epoch 1 train loss 0.0696482011762268
epoch 1 val loss 0.0516533007033556
epoch 2 train loss 0.06459392296134733
epoch 2 val loss 0.059315823400240456
epoch 3 train loss 0.06040172651378279
epoch 3 val loss 0.04591185389636237
epoch 4 train loss 0.057493147176913006
epoch 4 val loss 0.04242303346212485
epoch 5 train loss 0.054864771687373956
epoch 5 val loss 0.043910961222829795
epoch 6 train loss 0.052235604243315775
epoch 6 val loss 0.042071205454425574
epoch 7 train loss 0.05015943733939967
epoch 7 val loss 0.03606193821465016
epoch 8 train loss 0.04827413227846383
epoch 8 val loss 0.043114211312317054


KeyboardInterrupt: 

In [17]:
def evaluate_terminal(board):
    if board.is_checkmate():
        return -1.0 if board.turn == chess.WHITE else 1.0
    if board.is_stalemate() or board.is_insufficient_material():
        return None

def order_moves(board, legal_moves):
    def score(move):
        s = 0
        if board.is_capture(move):
            victim = board.piece_at(move.to_square)
            attacker = board.piece_at(move.from_square)
            if victim and attacker:
                s += 10 * PIECE_VALUES.get(victim.piece_type, 0) - PIECE_VALUES.get(attacker.piece_type, 0)
        if board.gives_check(move):
            s += 5
        return s
    return sorted(legal_moves, key=score, reverse=True)

def minimax(board, depth, alpha, beta, model, device):
    term = evaluate_terminal(board)
    if term is not None:
        return term
    if depth == 0:
        encoded = encode_board(board)
        heuristics = compute_heuristic_features(board)
        input_tensor = torch.tensor(encoded, dtype=torch.float32).unsqueeze(0).to(device)
        heur_tensor = torch.tensor(heuristics, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            net_value = model(input_tensor, heur_tensor).item()
        return net_value

    legal_moves = order_moves(board, list(board.legal_moves))

    if depth == 1:
        child_boards = []
        child_heuristics = []
        child_terms = []
        for move in legal_moves:
            board.push(move)
            t = evaluate_terminal(board)
            child_terms.append(t)
            if t is None:
                child_boards.append(encode_board(board))
                child_heuristics.append(compute_heuristic_features(board))
            board.pop()

        net_values = []
        if child_boards:
            batch_input = torch.tensor(np.stack(child_boards), dtype=torch.float32).to(device)
            batch_heuristics = torch.tensor(np.stack(child_heuristics), dtype=torch.float32).to(device)
            with torch.no_grad():
                net_values = model(batch_input, batch_heuristics).squeeze(1).cpu().numpy()
        net_iter = iter(net_values)

        values = [t if t is not None else next(net_iter) for t in child_terms]
        return max(values) if board.turn == chess.WHITE else min(values)

    if board.turn == chess.WHITE:
        best_val = -float('inf')
        for move in legal_moves:
            board.push(move)
            val = minimax(board, depth - 1, alpha, beta, model, device)
            board.pop()
            best_val = max(best_val, val)
            alpha = max(alpha, val)
            if beta <= alpha:
                break
        return best_val
    else:
        best_val = float('inf')
        for move in legal_moves:
            board.push(move)
            val = minimax(board, depth - 1, alpha, beta, model, device)
            board.pop()
            best_val = min(best_val, val)
            beta = min(beta, val)
            if beta <= alpha:
                break
        return best_val

def choose_best_move(board, model, device, depth=3):
    legal_moves = order_moves(board, list(board.legal_moves))
    best_move = None
    best_val = -float('inf') if board.turn == chess.WHITE else float('inf')
    alpha, beta = -float('inf'), float('inf')

    for move in legal_moves:
        board.push(move)
        val = minimax(board, depth - 1, alpha, beta, model, device)
        board.pop()

        if board.turn == chess.WHITE:
            if val > best_val:
                best_val, best_move = val, move
            alpha = max(alpha, val)
        else:
            if val < best_val:
                best_val, best_move = val, move
            beta = min(beta, val)

    return best_move, best_val


Code to Play Against the AI

In [ ]:
board=chess.Board()
model.load_state_dict(torch.load('best.pth')['model_state_dict'])
model.eval()
while not board.is_game_over():
  if board.turn==True:
    print(board)
    print("Legal moves:", [board.san(m) for m in board.legal_moves])
    move=input("Enter your move in Chess Notation: ")
    try:
      board.push_san(move)
    except:
      print("Illegal move, try again.")
  else:
      best_move, best_value = choose_best_move(board, model, device, depth=3)
      board.push(best_move)


print(board)
print("Game over", board.result())


In [11]:
def self_play_game(model, temperature=1.0, search_depth=1):
  model.eval()
  board = chess.Board()
  states=[]
  while not board.is_game_over():
    states.append((board.fen(), board.turn))
    legal_moves=list(board.legal_moves)
    current_turn=board.turn

    values = []
    for move in legal_moves:
        board.push(move)
        val = minimax(board, search_depth - 1, -float('inf'), float('inf'), model, device)
        board.pop()
        values.append(val)
    values = np.array(values, dtype=np.float32)
    values = values if current_turn == chess.WHITE else -values

    values_t = torch.tensor(values, dtype=torch.float32) / temperature
    probs = torch.softmax(values_t, dim=0)
    move_idx = torch.multinomial(probs, 1).item()
    best_move = legal_moves[move_idx]
    board.push(best_move)

  return states, board.result()

In [ ]:
model_old = ChessValueNet().to(device)
model_old.load_state_dict(torch.load('best.pth', map_location=device)['model_state_dict'])
model_old.eval()
training_data=[]
for run in range(300):
  states, results=self_play_game(model)
  if results == '1-0':
        target = 1.0
  elif results == '0-1':
        target = -1.0
  else:
        target = 0.0
  for fen, turn in states:
    board_obj = chess.Board(fen)
    encoded_version=encode_board(board_obj)
    heuristic_version=compute_heuristic_features(board_obj)
    training_data.append((encoded_version, heuristic_version, target))
x_train=torch.tensor(np.array([item[0] for item in training_data]), dtype=torch.float32)
h_train=torch.tensor(np.array([item[1] for item in training_data]), dtype=torch.float32)
y_train=torch.tensor(np.array([item[2] for item in training_data]), dtype=torch.float32).reshape(-1, 1)


In [12]:
def evaluation_between_models(model, model_old, num_games=20, search_depth=2):
    model.eval()
    model_old.eval()
    wins = 0

    for matches in range(num_games):
        board = chess.Board()
        challenger_is_white = (matches % 2 == 0)

        while not board.is_game_over():
            current_player = model if (board.turn == challenger_is_white) else model_old
            legal_moves = order_moves(board, list(board.legal_moves))
            if not legal_moves:
                break

            current_turn = board.turn
            best_move = None
            best_val = -float('inf') if current_turn == chess.WHITE else float('inf')
            for move in legal_moves:
                board.push(move)
                val = minimax(board, search_depth - 1, -float('inf'), float('inf'), current_player, device)
                board.pop()
                if current_turn == chess.WHITE:
                    if val > best_val:
                        best_val, best_move = val, move
                else:
                    if val < best_val:
                        best_val, best_move = val, move
            board.push(best_move)

        result = board.result()
        if (result == '1-0' and challenger_is_white) or (result == '0-1' and not challenger_is_white):
            wins += 1

    print(f"Evaluation finished. Challenger won {wins}/{num_games} games.")
    if wins > num_games * 0.55:
        print("New Champion, Updating model_old and saving best.pth")
        model_old.load_state_dict(model.state_dict())
        torch.save({'model_state_dict': model.state_dict()}, 'best.pth')
    else:
        print("Challenger did not improve enough.")


In [ ]:
model.train()

rl_dataset = torch.utils.data.TensorDataset(x_train, h_train, y_train)
rl_loader = DataLoader(rl_dataset, batch_size=64, shuffle=True)


rl_optimizer = optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-4)

for epoch in range(4):
    total_loss = 0.0
    for batch_x, batch_h, batch_y in rl_loader:
        batch_x, batch_h, batch_y = batch_x.to(device), batch_h.to(device), batch_y.to(device)
        rl_optimizer.zero_grad()
        predictions = model(batch_x, batch_h)
        loss = criterion(predictions, batch_y)
        loss.backward()
        rl_optimizer.step()
        total_loss += loss.item()

    if epoch % 1 == 0:
        print(f"RL Epoch {epoch}, Avg Loss: {total_loss / len(rl_loader):.4f}")

evaluation_between_models(model, model_old)


In [8]:
!pip install gradio

In [14]:
import gradio as gr
import chess.svg

model = ChessValueNet().to(device)
model.load_state_dict(torch.load('best.pth')['model_state_dict'])
model.eval()

def make_bot_move(board):
    if not list(board.legal_moves):
        return None
    best_move, best_value = choose_best_move(board, model, device, depth=3)
    return best_move

def get_legal_moves_str(board):
    return ", ".join([board.san(move) for move in board.legal_moves])


def new_game():
    global current_board
    current_board = chess.Board()
    legal_moves_str = get_legal_moves_str(current_board)

    return (chess.svg.board(current_board),
            "",
            legal_moves_str,
            gr.update(interactive=True),
            gr.update(value="", interactive=True))

def make_move(move_san):
    global current_board

    if current_board.is_game_over():
        return (chess.svg.board(current_board),
                f"Game Over! Result: {current_board.result()}",
                "Game Over",
                gr.update(interactive=False),
                gr.update(value="", interactive=False))

    try:
        player_move = current_board.parse_san(move_san)
        current_board.push(player_move)
        game_status = ""

        if current_board.is_game_over():
            game_status = f"Game Over! Result: {current_board.result()}"
            legal_moves_str = "Game Over"
            make_move_interactive = False
            move_input_interactive = False
        else:
            bot_move = make_bot_move(current_board)
            if bot_move:
                current_board.push(bot_move)
                if current_board.is_game_over():
                    game_status = f"Game Over! Result: {current_board.result()}"
                    legal_moves_str = "Game Over"
                    make_move_interactive = False
                    move_input_interactive = False
                else:
                    game_status = f"Bot moved: {current_board.san(bot_move)}"
                    legal_moves_str = get_legal_moves_str(current_board)
                    make_move_interactive = True
                    move_input_interactive = True
            else:
                game_status = "Bot has no legal moves."
                legal_moves_str = "Game Over"
                make_move_interactive = False
                move_input_interactive = False

        return (chess.svg.board(current_board),
                game_status,
                legal_moves_str,
                gr.update(interactive=make_move_interactive),
                gr.update(value="", interactive=move_input_interactive))

    except Exception:
        return (chess.svg.board(current_board),
                "Illegal move, try again.",
                get_legal_moves_str(current_board),
                gr.update(interactive=True),
                gr.update(value="", interactive=True))


In [18]:
current_board = chess.Board()

with gr.Blocks() as demo:
    gr.Markdown("# Chess Bot Interface")
    with gr.Row():
        with gr.Column(scale=2):
            board_display = gr.HTML(chess.svg.board(current_board))
            game_status_output = gr.Textbox(label="Game Status", interactive=False)
            legal_moves_output = gr.Textbox(label="Legal Moves", interactive=False, lines=3)
        with gr.Column(scale=1):
            move_input = gr.Textbox(label="Enter your move (e.g., 'e2e4')")
            make_move_button = gr.Button("Make Move")
            new_game_button = gr.Button("New Game")

    make_move_button.click(
        fn=make_move,
        inputs=[move_input],
        outputs=[board_display, game_status_output, legal_moves_output, make_move_button, move_input]
    )

    new_game_button.click(
        fn=new_game,
        inputs=[],
        outputs=[board_display, game_status_output, legal_moves_output, make_move_button, move_input]
    )

    demo.load(new_game, inputs=[], outputs=[board_display, game_status_output, legal_moves_output, make_move_button, move_input])

demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://cc4f864b20e7754663.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://cc4f864b20e7754663.gradio.live
